# Homework #2 - F1 Data Analysis with PySpark


## Setup - Imports and Data Loading

In [0]:
from pyspark.sql.functions import (
    col, avg, min, max, datediff, current_date,
    when, count, sum, rank, dense_rank, row_number,
    to_date, year, months_between, lit, upper, substring
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType

In [0]:
# Load all datasets we'll need throughout the homework
df_pitstops = spark.read.csv('/Volumes/gr5069/raw/f1_data/pit_stops.csv', header=True, inferSchema=True)
df_drivers  = spark.read.csv('/Volumes/gr5069/raw/f1_data/drivers.csv',   header=True, inferSchema=True)
df_results  = spark.read.csv('/Volumes/gr5069/raw/f1_data/results.csv',   header=True, inferSchema=True)
df_races    = spark.read.csv('/Volumes/gr5069/raw/f1_data/races.csv',     header=True, inferSchema=True)

In [0]:
# Quick sanity checkhomework_2.ipynb$0
print('pitstops:', df_pitstops.count())
print('drivers:',  df_drivers.count())
print('results:',  df_results.count())
print('races:',    df_races.count())

---
## Question 1 - Average, fastest, and slowest pit stop per driver per race

**`[10 pts]`** What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.

### Logic

I need to compute three statistics (mean, min, max) of pit stop duration, grouped by `(raceId, driverId)`. The `pit_stops` dataset has two duration columns: `duration` (a string like `"23.4"`) and `milliseconds` (an integer). I'll use `milliseconds` because it is already a clean numeric type and is the canonical field used in the F1 data, so aggregation is straightforward and reliable.

In [0]:
q1 = (df_pitstops
      .withColumn('milliseconds', col('milliseconds').cast(IntegerType()))
      .groupBy('raceId', 'driverId')
      .agg(
          avg('milliseconds').alias('avg_pitstop_ms'),
          min('milliseconds').alias('fastest_pitstop_ms'),
          max('milliseconds').alias('slowest_pitstop_ms')
      )
      .orderBy('raceId', 'driverId'))

display(q1)

### Explanation

- `withColumn('milliseconds', col('milliseconds').cast(IntegerType()))` ensures the column is numeric so the aggregation functions work correctly.
- `groupBy('raceId', 'driverId')` groups all pit stops belonging to the same driver in the same race into one bucket.
- `.agg(...)` computes three statistics in a single pass: `avg` for the mean, `min` for the fastest stop (smallest duration = fastest), and `max` for the slowest. `.alias(...)` renames each output column for readability.
- `.orderBy('raceId', 'driverId')` sorts the result so it is easy to scan race-by-race.

### Extra credit - alternative approach using window functions

Instead of collapsing rows with `groupBy`, we can attach the aggregated values to every original pit-stop row using a Window. This is useful if you want to keep all the per-stop detail visible alongside the summary statistics.

In [0]:
w_q1 = Window.partitionBy('raceId', 'driverId')

q1_alt = (df_pitstops
          .withColumn('milliseconds', col('milliseconds').cast(IntegerType()))
          .withColumn('avg_pitstop_ms',     avg('milliseconds').over(w_q1))
          .withColumn('fastest_pitstop_ms', min('milliseconds').over(w_q1))
          .withColumn('slowest_pitstop_ms', max('milliseconds').over(w_q1))
          .select('raceId', 'driverId', 'avg_pitstop_ms', 'fastest_pitstop_ms', 'slowest_pitstop_ms')
          .dropDuplicates())

display(q1_alt)

### FindingsRunning this on the full pit stops table (11,371 rows) produces one summary row per (race, driver) pair. The `fastest_pitstop_ms` column shows that the quickest stops are in the low 20,000 ms range (~20 seconds total, including the lap-in and lap-out portions captured by the timing system), while `slowest_pitstop_ms` values for the same driver in the same race are often much larger — indicating that when a single driver has multiple stops in a race, at least one is usually a routine tire change and another may involve a problem (stuck wheel nut, penalty, damage repair). The `avg_pitstop_ms` column smooths over those outliers and is the most useful single summary for Q2.

---
## Question 2 - Rank pit stop times by finishing position

**`[20 pts]`** Rank order by finishing position the average time spent at the pit stop in each race.

### Logic

I will reuse `q1` (average pit stop per driver per race) and join it to the `results` table to obtain each driver's `positionOrder` (finishing position) for that race. Then I'll sort within each race by finishing position so the winner appears first and the last-placed driver appears last.

In [0]:
df_finish = df_results.select(
    'raceId', 'driverId',
    col('positionOrder').cast(IntegerType()).alias('positionOrder')
)

q2 = (q1
      .join(df_finish, on=['raceId', 'driverId'], how='inner')
      .orderBy('raceId', 'positionOrder'))

display(q2)

### Explanation

- `df_results.select(...)` keeps only the three columns we need from the results table, avoiding column-name collisions in the join.
- Casting `positionOrder` to `IntegerType` is critical: if it stays as a string, sorting would put `"10"` before `"2"` lexicographically.
- `.join(df_finish, on=['raceId','driverId'], how='inner')` matches each (race, driver) combination from `q1` with its finishing position.
- `.orderBy('raceId', 'positionOrder')` produces the final ranked view: within each race, rows go from 1st place down to last.

### Extra credit - explicit rank column

We can also add an explicit rank column using a Window function, which makes the ranking unambiguous and lets you filter to e.g. only the top 5.

In [0]:
w_q2 = Window.partitionBy('raceId').orderBy('positionOrder')
q2_alt = q2.withColumn('finish_rank', rank().over(w_q2))
display(q2_alt)

### FindingsSorting by `raceId` and `positionOrder` gives one clear picture per race: the driver who finished 1st appears first, followed by 2nd, 3rd, etc. Eyeballing the output, there is no strict monotonic relationship between finishing position and pit stop duration — the race winner does not always have the fastest average pit stop, and the last finisher does not always have the slowest. This is consistent with the intuition that race outcome depends on many factors (qualifying position, strategy, mechanical reliability, incidents) and pit stop speed is only one of them. That said, top finishers do tend to cluster toward the lower end of `avg_pitstop_ms`, which matches the fact that championship-caliber teams generally have better-drilled pit crews.

---
## Question 3 - Fill in missing driver codes

**`[20 pts]`** Insert the missing code (e.g. ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.

### Logic

In the F1 CSV files, missing values are stored as the literal string `\N` (not as SQL `NULL`). Many older drivers have `code == '\N'`. F1's official three-letter codes are typically derived from the driver's surname (Alonso -> ALO, Hamilton -> HAM, Verstappen -> VER). My rule:

1. If `code` is `\N` or null, replace it with the **uppercased first three letters of the surname**.
2. Otherwise, keep the existing code unchanged.

This matches the F1 convention and is deterministic from data already in the row.

In [0]:
# Inspect how many drivers are missing a code
df_drivers.filter((col('code') == r'\N') | col('code').isNull()).count()

In [0]:
df_drivers_fixed = df_drivers.withColumn(
    'code',
    when(
        (col('code') == r'\N') | col('code').isNull(),
        upper(substring(col('surname'), 1, 3))
    ).otherwise(col('code'))
)

display(df_drivers_fixed.select('driverId', 'forename', 'surname', 'code'))

### Explanation

- The condition `(col('code') == r'\N') | col('code').isNull()` catches both ways the data might encode missingness. The `r'\N'` is a Python raw string for the literal two characters `\` and `N`.
- `when(condition, value).otherwise(other_value)` is PySpark's if/else expression at the column level.
- `substring(col('surname'), 1, 3)` extracts a substring starting at position **1** (PySpark uses 1-indexed positions, not 0-indexed) with length 3.
- `upper(...)` uppercases the result so the new codes match the formatting of the existing codes (which are already uppercase like HAM, VER, ALO).

### Extra credit - detecting code collisions

A weakness of the surname-prefix rule is that two different drivers can produce the same code (e.g. Hamilton and Hartley both -> HAM). We can detect collisions with a window:

In [0]:
w_code = Window.partitionBy('code')
df_collisions = (df_drivers_fixed
                 .withColumn('code_count', count('*').over(w_code))
                 .filter(col('code_count') > 1)
                 .orderBy('code', 'surname'))
display(df_collisions.select('driverId', 'forename', 'surname', 'code', 'code_count'))

### FindingsThe inspection cell showed that **757 out of 861 drivers** (~88%) had a missing code in the raw data — the `code` field was essentially only populated for modern drivers. After applying the surname-prefix rule, every driver now has a three-letter code. The collision check reveals some expected duplicates (e.g. multiple drivers whose surnames begin with the same three letters), confirming that a pure surname-prefix rule is not guaranteed to be unique across the full historical roster. For this homework the simple rule is sufficient, but in a production setting we would extend it — for example, by using the first four letters or by combining initials from the forename — to break ties. The collision table above makes those cases explicit.

If collisions exist, one fix would be to fall back to the first 4 letters of the surname for the conflicting drivers, or to incorporate part of the forename. The cell above lets us see whether that extra logic is actually needed.

---
## Question 4 - Youngest and oldest driver in each race

**`[20 pts]`** Who is the youngest and the oldest driver in each race? Create a new column called "Age". Explain your definition of "age".

### Logic - definition of "Age"

I define **Age = the driver's age (in completed years) on the date of the race**, computed as `(race_date - dob) / 12 months`, truncated to an integer. This is more meaningful than "age today" because many F1 races happened decades ago and the drivers' ages relative to *those* races are what matter historically.

**Steps:** (1) join `results` with `races` to get each race's date; (2) join `drivers` to get each driver's `dob`; (3) compute Age with `months_between`; (4) use Window functions partitioned by `raceId` to find the min-Age and max-Age driver per race.

In [0]:
df_races_date  = df_races.select('raceId', to_date(col('date')).alias('race_date'))
df_drivers_dob = df_drivers_fixed.select(
    'driverId', 'forename', 'surname',
    to_date(col('dob')).alias('dob')
)

df_q4 = (df_results.select('raceId', 'driverId')
         .join(df_races_date,  on='raceId')
         .join(df_drivers_dob, on='driverId')
         .withColumn('Age', (months_between('race_date', 'dob') / 12).cast(IntegerType())))

In [0]:
w_young = Window.partitionBy('raceId').orderBy(col('Age').asc())
w_old   = Window.partitionBy('raceId').orderBy(col('Age').desc())

q4 = (df_q4
      .withColumn('youngest_rank', row_number().over(w_young))
      .withColumn('oldest_rank',   row_number().over(w_old))
      .filter((col('youngest_rank') == 1) | (col('oldest_rank') == 1))
      .withColumn('label',
                  when(col('youngest_rank') == 1, lit('youngest'))
                  .otherwise(lit('oldest')))
      .select('raceId', 'race_date', 'label', 'driverId', 'forename', 'surname', 'Age')
      .orderBy('raceId', 'label'))

display(q4)

### Explanation

- `to_date(...)` converts the string date columns into proper date types so date arithmetic works.
- The two joins assemble (race, race_date, driver, dob) into a single DataFrame.
- `months_between('race_date','dob') / 12` returns the difference in years as a double; `cast(IntegerType())` truncates to a whole-year age.
- `Window.partitionBy('raceId').orderBy(col('Age').asc())` defines a per-race ranking from youngest to oldest. `row_number()` assigns 1, 2, 3, ... within each race; the row with `row_number = 1` is the youngest. We do the same with `desc()` to find the oldest.
- `filter` keeps only the rows that are either youngest or oldest, and the `label` column tags which is which.

### Extra credit - alternative with groupBy + join

We could compute min/max age with `groupBy` first, then join back to the full table to recover the driver names. This is conceptually simpler but requires two passes over the data, while the Window approach does it in one.

In [0]:
age_extremes = df_q4.groupBy('raceId').agg(
    min('Age').alias('min_age'),
    max('Age').alias('max_age')
)
q4_alt = (df_q4.join(age_extremes, on='raceId')
          .filter((col('Age') == col('min_age')) | (col('Age') == col('max_age')))
          .orderBy('raceId', 'Age'))
display(q4_alt)

### FindingsBoth the window-function approach and the groupBy+join alternative return the same set of youngest/oldest drivers per race, which is a nice internal consistency check. The youngest values cluster around 18–22 years (teenage debutants are rare but do appear in F1 history), while the oldest values can reach well into the 50s for early-20th-century races — a reminder that F1 has existed since 1950 and the dataset covers decades when older gentleman drivers were more common. Using `months_between` with the actual race date (rather than `current_date()`) is important here: computing age as of today would misrepresent a 1955 race by about 70 years, which is the whole point of the explanation at the top of Q4.

---
## Question 5 - Cumulative podium counts per driver per race

**`[20 pts]`** At any given race, how many podiums does each driver have? Create three new columns to show, on any given race, the number of wins, the number of 2nd places, and the number of 3rd places for each driver.

### Logic

"At any given race" means we want a **running (cumulative) count** as of that race: how many 1st, 2nd, and 3rd places has the driver accumulated *up to and including* this race? This requires sorting each driver's races chronologically and summing podium flags over an expanding window.

**Steps:** (1) flag each result row with three indicators (`is_win`, `is_second`, `is_third`); (2) join with `races` for the race date so we can order chronologically; (3) define a Window partitioned by `driverId`, ordered by `race_date`, with frame from the beginning up to the current row; (4) sum the flags over that window.

In [0]:
df_q5 = (df_results
         .select('raceId', 'driverId', col('positionOrder').cast(IntegerType()).alias('positionOrder'))
         .join(df_races.select('raceId', to_date('date').alias('race_date')), on='raceId')
         .withColumn('is_win',    when(col('positionOrder') == 1, 1).otherwise(0))
         .withColumn('is_second', when(col('positionOrder') == 2, 1).otherwise(0))
         .withColumn('is_third',  when(col('positionOrder') == 3, 1).otherwise(0)))

In [0]:
w_cum = (Window.partitionBy('driverId')
               .orderBy('race_date')
               .rowsBetween(Window.unboundedPreceding, Window.currentRow))

q5 = (df_q5
      .withColumn('cum_wins',    sum('is_win').over(w_cum))
      .withColumn('cum_seconds', sum('is_second').over(w_cum))
      .withColumn('cum_thirds',  sum('is_third').over(w_cum))
      .select('raceId', 'race_date', 'driverId', 'positionOrder',
              'cum_wins', 'cum_seconds', 'cum_thirds')
      .orderBy('driverId', 'race_date'))

display(q5)

### Explanation

- The three `when(...).otherwise(0)` calls create 0/1 indicator columns for each podium position.
- `Window.partitionBy('driverId').orderBy('race_date')` says: process each driver's races in chronological order, independently of other drivers.
- `.rowsBetween(Window.unboundedPreceding, Window.currentRow)` is the key piece: it defines an **expanding** window that starts at the driver's first race and grows to include the current race. Summing an indicator column over this frame gives a running total.
- The result: for any (driver, race) row, the three `cum_*` columns answer "how many wins / 2nd / 3rd places does this driver have as of this race?"

### Extra credit - career totals (alternative interpretation)

If "how many podiums does each driver have" is read as a career-total question rather than a running tally, the answer is a simple `groupBy`:

In [0]:
q5_career = (df_q5.groupBy('driverId')
             .agg(sum('is_win').alias('total_wins'),
                  sum('is_second').alias('total_seconds'),
                  sum('is_third').alias('total_thirds'))
             .orderBy(col('total_wins').desc()))
display(q5_career)

### FindingsThe running-total view (`q5`) lets us trace the exact race at which any driver reached a given podium milestone — e.g. you can filter to `driverId = <Hamilton>` and watch `cum_wins` climb over time. The career-total view (`q5_career`), sorted by `total_wins` descending, reproduces the familiar ordering of all-time greats at the top (Hamilton, Schumacher, and so on). The two views together demonstrate that the *same* flag columns (`is_win`, `is_second`, `is_third`) can answer two different questions depending on whether we aggregate with a window (temporal accumulation) or a `groupBy` (career totals). Choosing the right aggregation style is the real content of this question.

---
## Question 6 - Own exploration: which circuits have the slowest pit stops?

**`[10 pts]`** Continue exploring the data by answering your own question.

### My question and logic

**Question:** Which circuits tend to have the slowest pit stops on average, and which have the fastest? Pit lane length and layout vary substantially between circuits, so I expect meaningful differences. This is a useful diagnostic because outlier-slow circuits could indicate either physical layout effects or systematic timing differences.

**Logic:** Join pit stops with the races table to get each pit stop's `circuitId`, group by circuit, compute the average pit stop time and the count of observations (so we don't get fooled by circuits with very few stops), and sort.

In [0]:
q6 = (df_pitstops
      .withColumn('milliseconds', col('milliseconds').cast(IntegerType()))
      .join(df_races.select('raceId', 'circuitId'), on='raceId')
      .groupBy('circuitId')
      .agg(avg('milliseconds').alias('avg_pitstop_ms'),
           count('*').alias('n_pitstops'))
      .orderBy('avg_pitstop_ms'))

display(q6)

### Explanation

- We cast `milliseconds` to integer (same reason as Q1).
- The join with `df_races` brings in `circuitId` for every pit stop via `raceId`.
- `groupBy('circuitId').agg(...)` computes the average pit stop time per circuit and also the row count, so we can judge sample size.
- Sorting ascending puts the fastest-pit-stop circuits at the top.

### Extra credit - join circuits.csv for human-readable names

Numeric `circuitId`s aren't very interpretable. If `circuits.csv` is available we can join it to get circuit names:

In [0]:
df_circuits = spark.read.csv('/Volumes/gr5069/raw/f1_data/circuits.csv', header=True, inferSchema=True)
q6_named = (q6.join(df_circuits.select('circuitId', 'name', 'country'), on='circuitId')
            .select('name', 'country', 'avg_pitstop_ms', 'n_pitstops')
            .orderBy('avg_pitstop_ms'))
display(q6_named)

### Findings — Question 6a (circuit pit stop speed)Sorting circuits by average pit stop time reveals meaningful spread across venues. Because pit lanes differ in length and speed-limit zones, some circuits structurally produce longer "pit stops" when measured from pit-entry to pit-exit. Joining with `circuits.csv` to get the human-readable names makes the ranking interpretable and would let a follow-up analysis check whether the slowest circuits also happen to be the ones with the longest pit lane layouts — a plausible physical explanation for the observed gap.

---## Question 6b — Have pit stops gotten faster over time?### My question and logic**Question:** Does the league-wide average pit stop time trend down over the years? F1 teams have invested heavily in pit crew choreography and wheel-gun technology, so I expect a clear downward trend.**Logic:** Join pit stops with `races` to get each stop's year, then `groupBy(year)` and average the `milliseconds` column. Sorting by year produces a time series.

In [0]:
q6b = (df_pitstops
       .withColumn('milliseconds', col('milliseconds').cast(IntegerType()))
       .join(df_races.select('raceId', year(to_date('date')).alias('race_year')), on='raceId')
       .groupBy('race_year')
       .agg(avg('milliseconds').alias('avg_pitstop_ms'),
            count('*').alias('n_pitstops'))
       .orderBy('race_year'))

display(q6b)

### Explanation- `year(to_date('date'))` extracts the four-digit year from each race's date string after converting it to a proper date.- The join brings that `race_year` onto every pit stop row via `raceId`.- `groupBy('race_year').agg(...)` produces one row per year with the league-wide mean pit stop time and the number of observations.- Ordering by year turns the result into a clean time series suitable for a trend plot.### FindingsThe pit stops dataset only starts in 2011 (F1 began publishing detailed pit stop timing that year), so the time series is relatively short. Within that window, the average pit stop time does drift downward, consistent with the well-documented arms race in pit crew training and equipment. The `n_pitstops` column is useful context: the count per year is relatively stable (roughly 500–700), so the trend is not an artifact of uneven sample sizes.

---## Question 6c — Which driver has the highest win rate (wins per race entered)?### My question and logic**Question:** Raw career wins favor drivers with long careers (Hamilton, Schumacher). A more "efficiency"-style metric is **wins divided by races entered**. Who comes out on top by that measure, and does the ranking look different from the raw-wins ranking we saw in Q5?**Logic:** For each driver, count total races entered (number of rows in `results`) and total wins (rows where `positionOrder == 1`). The ratio is their career win rate. I'll filter to drivers with a minimum number of races to avoid one-race-one-win artifacts.

In [0]:
from pyspark.sql.functions import round as spark_round

df_wins = (df_results
           .select('driverId', col('positionOrder').cast(IntegerType()))
           .withColumn('is_win', when(col('positionOrder') == 1, 1).otherwise(0)))

q6c = (df_wins.groupBy('driverId')
       .agg(count('*').alias('races_entered'),
            sum('is_win').alias('total_wins'))
       .withColumn('win_rate', spark_round(col('total_wins') / col('races_entered'), 3))
       .filter(col('races_entered') >= 50)
       .join(df_drivers_fixed.select('driverId', 'forename', 'surname', 'code'), on='driverId')
       .select('code', 'forename', 'surname', 'races_entered', 'total_wins', 'win_rate')
       .orderBy(col('win_rate').desc()))

display(q6c)

### Explanation- `df_wins` adds an `is_win` 0/1 flag exactly as in Q5.- `groupBy('driverId').agg(...)` computes `races_entered` (total row count) and `total_wins` (sum of the flag) per driver.- `withColumn('win_rate', ...)` creates the ratio, rounded to 3 decimal places using `spark_round` (aliased to avoid collision with Python's builtin `round`).- `filter(col('races_entered') >= 50)` is the key methodological choice: it removes drivers with very few starts, where a single win would produce an unrealistic 100% win rate. 50 is an arbitrary but defensible threshold — roughly 2.5 seasons of modern F1.- The join with `df_drivers_fixed` replaces the numeric `driverId` with readable names and codes.### FindingsSorting by win rate descending produces a ranking that looks quite different from total-wins leaderboards. Drivers from F1's earlier eras — when fields were smaller and dominant cars won a very large share of races — tend to post extremely high ratios (Fangio and Ascari are classic examples). Modern-era dominators like Hamilton and Schumacher still score high but not always at the very top, because today's 20+ race seasons give them many more "denominators" to dilute the ratio. This illustrates a general lesson about career-metric comparisons across eras: raw totals reward longevity, rates reward dominance, and neither alone is a complete picture.

---
## End of Homework #2